In [ ]:
# 2
import numpy as np                                                      # anything with numbers
import pandas as pd                                                     # anything with tables
import xarray as xr                                                     # geospatial data
import matplotlib.pyplot as plt                                         # make visuals
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from workshop_utils import PROCESSED_DIR                                # PROCESSED_DIR is the data/processed/ directory, where the data we need is stored

print("Ready!")

## *PART III: Land Use and Land Use Change*

Köppen designed his climate thresholds around vegetation zones; now let's flip that around and classify *land cover itself* directly from satellite imagery — using the exact same recipe as Part I's homemade climate classifier: turn a decision rule into a `def` function, apply it to a whole grid at once, and map the result.

Part II already showed that the raw, single-date Sentinel-2 tiles for Negros are cloudy and full of gaps — not something you'd want to build a classifier on. So instead we'll work directly with the properly-engineered data: `negros_imagery.nc`, the cloud-masked, season-matched **composite** imagery from the end of Part II — a historical composite (Landsat 5, pooled from 1988-2000) and a recent one (Sentinel-2, pooled from 2022-2024).


In [ ]:
img = xr.open_dataset(PROCESSED_DIR / "day2" / "negros_imagery.nc")
print(img.time.values)

for label, t in [("1996 (historical)", "1996-03-15"), ("2023 (recent)", "2023-03-15")]:
    nan_frac = float(np.isnan(img["red"].sel(time=t).values).mean())
    print(f"{label}: {nan_frac:.1%} no-data")

> 💬 **Discussion:** The recent (2023) composite is essentially gap-free — 0.0% no-data. The historical (1996) composite is a different story: **~31% no-data**, worse than you might expect given it's pooled from 133 scenes. Pooling *more* scenes doesn't automatically mean *fewer* gaps — it means more honest bookkeeping about where *no* scene in 1988-2000 ever caught a clear view. Older imagery is scarcer and noisier, full stop; properly accounting for missing data doesn't make it disappear, it just stops you from pretending it isn't there.

Let's look at both composites as true-colour images:

In [ ]:
def to_rgb(ds_slice):
    r = np.nan_to_num(ds_slice["visual_r"].values)
    g = np.nan_to_num(ds_slice["visual_g"].values)
    b = np.nan_to_num(ds_slice["visual_b"].values)
    return np.dstack([r, g, b]).astype(np.uint8)


rgb_1996 = to_rgb(img.sel(time="1996-03-15"))
rgb_2023 = to_rgb(img.sel(time="2023-03-15"))

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].imshow(rgb_1996)
axes[0].set_title("~1996 (Landsat 5, 133 pooled scenes)")
axes[1].imshow(rgb_2023)
axes[1].set_title("~2023 (Sentinel-2, 156 pooled scenes)")
for ax in axes:
    ax.axis("off")
fig.suptitle("Negros, cloud-free composite true colour — historical vs. recent")

In [ ]:
landsat = img.sel(time="2023-03-15")
landsat.visual_r.plot()

> 💬 **Discussion:** Compare these to the single-date Sentinel-2 images from Part II — no cloud streaks, no dark data-gap patches, consistent colour balance across the whole scene. This is what the extra engineering (pooling scenes, masking clouds, taking the median) buys you.

---

### 1) NDVI: turning two bands into one meaningful number

**Healthy, lush-green vegetation absorbs a lot of red light but reflects a lot of near-infrared**. We can combine the `red` and `nir` bands in the [Normalized Difference Vegetation Index NDVI](https://en.wikipedia.org/wiki/Normalized_difference_vegetation_index), the most widely used metric to derive vegetation status from remote sensing data. NDVI is computed as:

**NDVI = (NIR − Red) / (NIR + Red)**

NDVI ranges from -1 to +1: strongly **negative** for water, **near zero** for bare soil/urban surfaces, and **high** (often >0.3) for healthy vegetation. Let's write a small reusable function for it:

In [ ]:
def compute_ndvi(red, nir):
    return (nir - red) / (nir + red)


# --- Apply it to the 2023 (recent) composite and map it ---
recent = img.sel(time="2023-03-15")
ndvi_2023 = compute_ndvi(recent["red"], recent["nir"])

fig, ax = plt.subplots(figsize=(7, 7))
ndvi_2023.plot(ax=ax, cmap="RdYlGn", vmin=-1, vmax=1, cbar_kwargs={"label": "NDVI"})
ax.set_title("NDVI — Negros, 2023 composite")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Which parts of the map are reddish (low/negative NDVI)? Which are green (high NDVI)? Do the reddish areas match what you'd expect for water and urban areas from the true-colour image above?

---

### 2) From NDVI to land-cover classes

Let's turn NDVI into actual land-cover classes using a simple, interpretable rule — deliberately similar in spirit to Köppen's group logic: a handful of threshold conditions on physically meaningful numbers.

Looking at real values in this composite: 2023's NDVI ranges roughly -0.56 to +0.91 (mean ~0.31); 1996 runs a bit hotter, mean ~0.47. *Brightness* (the average of red and nir) is much lower over water than over land or bare ground. `negros_imagery.nc` stores physical **reflectance** (0-1 float), not raw digital numbers — so our brightness threshold has to live on that scale too. That gives us three classes:

| Class | Rule |
|---|---|
| **Water** | low brightness *and* low NDVI (dark in both bands — water absorbs rather than reflects) |
| **Vegetation** | high NDVI |
| **Bare/Urban** | everything else (moderate-to-high brightness, low NDVI) |

We'll use `brightness < 0.08` and `NDVI < 0.1` for Water, and `NDVI > 0.3` for Vegetation — thresholds picked by eye against this scene's own numbers.

### 🖊️ Your turn: `classify_landcover_2D`

Reuse the exact "boolean-mask, reverse-priority overwrite" trick from `classify_koppen_2D` in Part I — same technique, brand new domain, xarray structure preserved throughout.

In [ ]:
def classify_landcover_2D(red, nir, brightness_thr=0.08):

    ndvi = compute_ndvi(red, nir).clip(-1, 1)  # a few pixels have near-zero red+nir -> guard the ratio
    brightness = (red + nir) / 2

    landcover = xr.full_like(red, "Bare/Urban", dtype="<U10")               # ✏️ default/"else" case
    landcover.values[(brightness < brightness_thr).values & (ndvi < 0.1).values] = "Water"   # ✏️ dark in both bands
    landcover.values[(ndvi > 0.3).values] = "Vegetation"                               # ✏️ high NDVI, checked last -> wins
    landcover.values[np.isnan(red.values)] = ""                                        # ✏️ cloud/no-data gaps - not a real class

    return landcover


# --- Try it on the 2023 composite ---
landcover_2023 = classify_landcover_2D(recent["red"], recent["nir"])
print(dict(zip(*np.unique(landcover_2023.values, return_counts=True))))

Now let's map it, using its own small categorical colour scheme — same recipe as the Köppen-Geiger maps:

In [ ]:
LC_INFO = [
    (1, "Water",      "#3B4CC0"),
    (2, "Vegetation", "#2E8B57"),
    (3, "Bare/Urban", "#C2A878"),
]
lc_cmap = mcolors.ListedColormap([c for _, _, c in LC_INFO])
lc_norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, len(LC_INFO) + 1.5), ncolors=len(LC_INFO))
LC_TO_CODE = {name: code for code, name, _ in LC_INFO}


def landcover_to_code(landcover):
    # 💡 same trick as class_to_code in Part I: map each string label to its numeric code, keep the DataArray/coords intact
    code_grid = xr.full_like(landcover, np.nan, dtype="float64")
    for name, code in LC_TO_CODE.items():
        code_grid.values[(landcover == name).values] = code
    return code_grid


fig, ax = plt.subplots(figsize=(7, 7))
code_2023 = landcover_to_code(landcover_2023)
ax.pcolormesh(code_2023.longitude, code_2023.latitude, code_2023, cmap=lc_cmap, norm=lc_norm)
ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
ax.set_title("Homemade land cover — Negros, 2023 composite")
ax.set_aspect("equal")
patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
ax.legend(handles=patches, loc="lower left", frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Compare this to the true-colour image above — does Water/Vegetation/Bare-Urban line up with what you saw by eye?

Let's also sanity-check this against the real `treecover2000` map from Part II — side by side, purely by eye (different sensor, different resolution, so no pixel-by-pixel score, same design choice as Part I Step 7):

In [ ]:
# author comment: also from Part II
forest_ds = xr.open_dataset("../data/processed/day2/negros_forest_change.nc")
treecover = forest_ds["treecover2000"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

axes[0].pcolormesh(code_2023.longitude, code_2023.latitude, code_2023, cmap=lc_cmap, norm=lc_norm)
axes[0].set_title("Our homemade land cover (2023 composite)")

treecover.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Hansen tree cover (2000)")

for ax in axes:
    ax.set_aspect("equal")

> 💬 **Discussion:** Working with satellite data is clearly a tricky business, even with good composite imagery — but this already looks a lot more trustworthy than a single cloudy scene would. Let's push further: does our classifier hold up if we apply it to the *other* composite date too?

---

### 3) Classify both dates

We already loaded both composite dates as `img` above — no need to reopen anything, just apply `classify_landcover_2D` to each date's `red`/`nir`:

In [ ]:
historical = img.sel(time="1996-03-15")

landcover_1996 = classify_landcover_2D(historical["red"], historical["nir"])
# landcover_2023 already computed above

print("1996:", dict(zip(*np.unique(landcover_1996.values, return_counts=True))))
print("2023:", dict(zip(*np.unique(landcover_2023.values, return_counts=True))))

Map both side by side:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

for ax, landcover, title in zip(axes, [landcover_1996, landcover_2023], ["1996", "2023"]):
    code_grid = landcover_to_code(landcover)                                                        # ✏️ landcover_to_code(landcover)
    ax.pcolormesh(code_grid.longitude, code_grid.latitude, code_grid, cmap=lc_cmap, norm=lc_norm)   # ✏️ pcolormesh with lc_cmap/lc_norm
    ax.scatter(122.95, 10.68, color="black", s=30, zorder=5)                                        # ✏️ mark Bacolod
    ax.set_title(title)
    ax.set_aspect("equal")

patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
fig.legend(handles=patches, loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Homemade Land Cover — Negros, 1996 vs. 2023")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

> 💬 **Discussion:** Look at the **Water** counts: they jump from under **1%** of the scene in 1996 to over **40%** in 2023, using the exact same rule. Negros' coastline didn't move — this is Landsat 5 and Sentinel-2 simply not being perfectly cross-calibrated in absolute brightness, a real, known challenge when mixing sensors across decades. Could we fix that, at least partially?

---

### 4) Calibrating brightness across sensors, using the ocean

Here's an idea: **deep ocean water reflects light in a fairly consistent way, regardless of which satellite is looking at it** — open ocean is dark and spectrally flat in the red/NIR, with no vegetation, no buildings, and no seasonal canopy changes to confuse things. That makes ocean pixels a natural *reference target*: if the same patch of ocean reads much brighter in one composite than the other, that gap is a property of the sensor/processing, not of the water itself. We can use it to rescale one sensor onto the other's brightness scale.

We already have a **land-sea mask** at ~1km resolution from Day 1/2 (`land-sea-mask_0p00833333.nc`, 1 = land, 0 = ocean). Let's put it onto the composite's own (finer) grid and use it to pull out ocean-only pixels:

In [ ]:
lsm = xr.open_dataarray(PROCESSED_DIR / "land-sea-mask_0p00833333.nc").sel(
    lon=slice(float(img.longitude.min()), float(img.longitude.max())),
    lat=slice(float(img.latitude.max()), float(img.latitude.min())),
)
# ✏️ interpolate the coarser land-sea mask onto img's own grid, nearest-neighbour so it stays binary (0/1)
ocean = lsm.rename({"lat": "latitude", "lon": "longitude"}).interp(
    latitude=img.latitude, longitude=img.longitude, method="nearest"
) == 0

print(f"Ocean pixels: {float(ocean.mean()):.1%} of the scene")

Now compare the *mean* red and NIR reflectance over ocean pixels only, for each date:

In [ ]:
def ocean_mean(band):
    values = band.values
    return float(values[ocean.values & ~np.isnan(values)].mean())


red_1996, nir_1996 = historical["red"], historical["nir"]      # ✏️ historical["red"], historical["nir"]
red_2023, nir_2023 = recent["red"], recent["nir"]               # ✏️ recent["red"], recent["nir"]

print(f"Ocean red — 1996: {ocean_mean(red_1996):.3f}   2023: {ocean_mean(red_2023):.3f}")
print(f"Ocean nir — 1996: {ocean_mean(nir_1996):.3f}   2023: {ocean_mean(nir_2023):.3f}")

> 💬 **Discussion:** Same ocean, roughly **4x brighter** in the 1996 (Landsat) composite than in 2023 (Sentinel-2) — in both bands. That's exactly the kind of offset that was swamping our Water classification. Let's use the ratio between these two numbers as a per-band **scale factor**, and rescale 1996 onto 2023's brightness scale:

In [ ]:
scale_red = ocean_mean(red_2023) / ocean_mean(red_1996)      # ✏️ how much dimmer is 2023's ocean, per band?
scale_nir = ocean_mean(nir_2023) / ocean_mean(nir_1996)
print(f"scale_red = {scale_red:.3f}   scale_nir = {scale_nir:.3f}")

red_1996_calibrated = red_1996 * scale_red                   # ✏️ rescale the *whole* 1996 scene, not just the ocean
nir_1996_calibrated = nir_1996 * scale_nir

Reclassify the calibrated 1996 scene with the exact same function and threshold, and compare all three:

In [ ]:
landcover_1996_calibrated = classify_landcover_2D(red_1996_calibrated, nir_1996_calibrated)

print("1996, raw:        ", dict(zip(*np.unique(landcover_1996.values, return_counts=True))))
print("1996, calibrated: ", dict(zip(*np.unique(landcover_1996_calibrated.values, return_counts=True))))
print("2023:             ", dict(zip(*np.unique(landcover_2023.values, return_counts=True))))

> 💬 **Discussion:** Calibration moves Water from **0.9%** of the scene to **8.1%** — nearly an order of magnitude closer to 2023's **40.1%**, using nothing but a single number per band. It doesn't fully close the gap: a single global scale factor can't correct for everything that differs between a 1988-2000 Landsat 5 composite and a 2022-2024 Sentinel-2 one — different atmospheric correction, sun-angle/viewing-angle effects, and each sensor's own spectral response curve all leave a residual. (The historical ocean samples are also noisier than the recent ones — a hint of leftover haze even in a "cloud-free" composite.) **Vegetation, meanwhile, barely moves** (58.3% raw → 57.0% calibrated) — the same robustness-to-brightness we noted above, now backed by an actual before/after number instead of just "nudge the threshold and see." That's why we'll keep trusting NDVI/Vegetation for change detection below, and treat Water/Bare-Urban as a lesson in how tricky cross-sensor comparisons really are.

---

You just:
- Learned what NDVI is and why it works (red absorption vs. NIR reflection in healthy vegetation)
- Reused Part I's exact "rule -> vectorized function -> map" pattern in a brand new domain
- Classified real satellite imagery into land-cover classes, for two points in time, and discovered why comparing brightness-based classes across two different sensors is risky
- Used ocean pixels as a physical reference point to calibrate brightness between Landsat and Sentinel-2 — and saw why that fix helps a lot without being perfect

Next: **Land Use Change**, where we put 1996 and 2023 side by side to actually measure what changed.

## Land Use Change

Two independent looks at land-use change on Negros, over different but overlapping periods: Hansen's own annual forest-loss labels (2001-2023), and change detected from *our own* homemade land-cover classifier (roughly 1996 vs. 2023) — cross-checked against each other, the same way Part I cross-checked its homemade climate map against Beck et al.

**One honest caveat before we start:** these aren't the same window. Sentinel-2 only launched in 2015, so reaching back to 1996 for imagery means switching sensors entirely (which is exactly why we needed the calibration step above) — this composite spans roughly 1996-2023, not Hansen's 2001-2023. We're not measuring the identical thing twice; we're making two independently-engineered measurements and seeing whether the two stories broadly agree.

### 1) Forest cover loss

`negros_forest_change.nc` has a second variable we haven't used yet: `lossyear` — the year each pixel first lost forest cover (1 = 2001, ..., 23 = 2023, 0 = no loss detected). Let's load it:

In [ ]:
lossyear = forest_ds["lossyear"]        # already opened above as forest_ds
lossyear

### 🖊️ How many pixels were lost each year?

`np.unique(..., return_counts=True)` is the numpy equivalent of pandas' `.value_counts()` from Part II, applied to a plain array:

In [ ]:
years, counts = np.unique(lossyear.values, return_counts=True)     # ✏️ np.unique(..., return_counts=True)

for year_code, count in zip(years, counts):
    if year_code == 0:
        continue                        # 0 = no loss detected, not an actual year
    print(f"{2000 + int(year_code)}: {count} pixels")

**Bar plot** of pixels lost per year:

In [ ]:
loss_only = counts[years != 0]
years_only = 2000 + years[years != 0].astype(int)

fig, ax = plt.subplots(figsize=(11, 4))

ax.bar(years_only, loss_only, color="firebrick")           # ✏️ years_only, loss_only

ax.set_title("Forest Loss per Year — Negros (Hansen et al.)")
ax.set_xlabel("Year")
ax.set_ylabel("Pixels lost (30m)")

> 💬 **Discussion:** Any standout years? *(Real data shows a clear spike around 2018, and again in 2022-2023 — any guesses why, before we tell you? Think El Niño-driven dry seasons/fires, logging booms, or just natural year-to-year noise.)*

Map `lossyear` spatially (masking the "no loss" pixels so they don't dominate the colour scale), next to `treecover2000`:

In [ ]:
lost_any = (lossyear > 0).astype("int8")   # 1 = forest lost at least once, 0 = no loss

fig, axes = plt.subplots(1, 2, figsize=(13, 8))

lsm_negros = xr.open_dataarray(PROCESSED_DIR / "land-sea-mask_0p00833333.nc").sel(
    lon=slice(float(treecover.longitude.min()), float(treecover.longitude.max())),
    lat=slice(float(treecover.latitude.max()), float(treecover.latitude.min())),
)

cmap_bin = mcolors.ListedColormap(["white", "red"])  # 0 -> white, 1 -> red
norm_bin = mcolors.BoundaryNorm([-0.5, 0.5, 1.5], cmap_bin.N)

lost_any.plot(
    ax=axes[0],
    cmap=cmap_bin,
    norm=norm_bin,
    cbar_kwargs={"ticks": [0, 1], "label": "Forest loss (0/1)"},
)
axes[0].set_title("Forest loss: binary")

treecover.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Tree cover (2000)")

for ax in axes:
    ax.contour(
        lsm_negros.lon, lsm_negros.lat, lsm_negros.values,
        levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7
    )
    ax.set_xlim(float(treecover.longitude.min()), float(treecover.longitude.max()))
    ax.set_ylim(float(treecover.latitude.min()), float(treecover.latitude.max()))
    ax.set_aspect("equal")

> 💬 **Discussion:** Where has loss concentrated? Does it look random, or clustered along certain patterns (roads, forest edges, specific regions)?

### 🖊️ Find the biggest hotspot

Let's turn "where has loss concentrated" into an actual number. We can reuse the same `.coarsen()` trick from Part II (where you coarsened `treecover` to make it faster to plot): sum the binary loss flag inside blocks of pixels, and find the block with the highest count.

In [ ]:
factor = 80  # ✏️ block size in pixels (~2.2 km x 2.2 km at this ~28m resolution)
loss_grid = lost_any.coarsen(latitude=factor, longitude=factor, boundary="trim").sum()

flat_idx = int(np.argmax(loss_grid.data))              # ✏️ np.argmax on the flattened grid
iy, ix = np.unravel_index(flat_idx, loss_grid.shape)    # ✏️ turn that back into a (row, col) pair
hotspot_lat = float(loss_grid.latitude[iy])
hotspot_lon = float(loss_grid.longitude[ix])

print(f"Biggest hotspot: {int(loss_grid.data[iy, ix])} lost pixels around ({hotspot_lat:.3f}, {hotspot_lon:.3f})")

That's one ~2 km block where **nearly 40%** of all 30 m pixels lost forest at some point since 2001 — the single most concentrated clearing event on the island in this dataset. Let's zoom the composite imagery in on it and see whether we can actually *see* that in the satellite pictures.

In [ ]:
half = 0.06   # ✏️ degrees of padding around the hotspot, ~13 km box
lat_lo, lat_hi = hotspot_lat - half, hotspot_lat + half
lon_lo, lon_hi = hotspot_lon - half, hotspot_lon + half

hist_zoom = img.sel(time="1996-03-15", latitude=slice(lat_hi, lat_lo), longitude=slice(lon_lo, lon_hi))
rec_zoom  = img.sel(time="2023-03-15", latitude=slice(lat_hi, lat_lo), longitude=slice(lon_lo, lon_hi))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(to_rgb(hist_zoom), extent=[lon_lo, lon_hi, lat_lo, lat_hi])
axes[0].set_title("1996 (Landsat), zoomed to hotspot")
axes[1].imshow(to_rgb(rec_zoom), extent=[lon_lo, lon_hi, lat_lo, lat_hi])
axes[1].set_title("2023 (Sentinel-2), zoomed to hotspot")
for ax in axes:
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Does either panel look more "cleared" than the other? Keep in mind the 1996 panel is that same darker, hazier Landsat 5 composite from earlier — the same colour-balance gap that inflated the Water class. Colour alone makes this a hard call. Let's set true-colour aside and check what our *classifier* says instead, since NDVI is supposed to be robust to exactly this kind of brightness mismatch — we already have `landcover_1996_calibrated` and `landcover_2023` computed for the whole island, so we just need to zoom into the same box:

In [ ]:
lc_hist_zoom = landcover_1996_calibrated.sel(latitude=slice(lat_hi, lat_lo), longitude=slice(lon_lo, lon_hi))
lc_rec_zoom  = landcover_2023.sel(latitude=slice(lat_hi, lat_lo), longitude=slice(lon_lo, lon_hi))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, lc, title in zip(axes, [lc_hist_zoom, lc_rec_zoom], ["1996 (calibrated)", "2023"]):
    code_grid = landcover_to_code(lc)
    ax.pcolormesh(code_grid.longitude, code_grid.latitude, code_grid, cmap=lc_cmap, norm=lc_norm)
    ax.set_title(title)
    ax.set_aspect("equal")
patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
fig.legend(handles=patches, loc="lower center", ncol=3, frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

for label, lc in [("1996", lc_hist_zoom), ("2023", lc_rec_zoom)]:
    veg_frac = float((lc == "Vegetation").mean())
    print(f"{label}: {veg_frac:.1%} classified as Vegetation in this hotspot")

> 💬 **Discussion:** Our classifier says **99.7%** Vegetation in 1996 and **99.3%** in 2023 — almost no change at all, in the single biggest forest-loss hotspot on the island! What's going on?

`lossyear` tracks *tree canopy* disturbance specifically, using a dedicated algorithm built from a dense annual time series. Our classifier only asks "is NDVI high?" — it can't tell primary forest apart from any other green cover. If cleared land here was replanted or regrew into farmland, tree plantations, or fast secondary vegetation (all common outcomes on Negros) rather than staying bare, it still reads as "Vegetation" to a simple two-class rule, even though Hansen's algorithm correctly flagged real forest loss underneath. That's a genuine, honest limitation — and it's a big part of why our own vegetation-loss estimate below (~2.7%) comes in under Hansen's (~3.5%): cases like this one, where forest becomes some *other* kind of green, are invisible to our classifier even though they're real deforestation.



### 🖊️ Quantify loss over the full 2001-2023 record

This is the record we'll cross-check our own composite-based classifier against in subsection 2, since our composite spans roughly 1996-2023:

In [ ]:
had_forest_2000 = treecover > 25                        # ✏️ treecover > 25 (a simple "forest" baseline threshold)
lost_full_record = (lossyear >= 1) & (lossyear <= 23)   # ✏️ lossyear codes 1-23 = calendar years 2001-2023

frac_lost_hansen_full = float((lost_full_record & had_forest_2000).sum()) / float(had_forest_2000.sum())
print(f"Fraction of Negros' 2000 forest baseline lost between 2001-2023 (Hansen): {frac_lost_hansen_full * 100:.1f}%")

---

### 2) Change detection from our own classifier

We already have `landcover_1996_calibrated` and `landcover_2023` from Part III. Let's write one reusable function to detect any transition between classes — general enough to answer "where did we lose vegetation?":

In [ ]:
def detect_change(before, after, from_class=None, to_class=None):
    # ✏️ from_class/to_class are optional: None means "any class". A pixel counts as changed if
    # it was `from_class` before, is `to_class` after, and the two dates actually differ.
    has_data  = (before != "") & (after != "")            # ✏️ ignore cloud/no-data gaps in either year!
    was_from  = True if from_class is None else (before == from_class)
    became_to = True if to_class   is None else (after == to_class)
    return has_data & was_from & became_to & (before != after)


lost_vegetation = detect_change(landcover_1996_calibrated, landcover_2023, from_class="Vegetation")
print(f"Pixels that were Vegetation in 1996 but aren't anymore: {int(lost_vegetation.sum())}")

💡 Note: without the `has_data` check, the huge no-data-gap difference between the two dates (~31% of the 1996 composite vs. 0% in 2023) would get counted as spurious "change" — always account for missing data before comparing two time periods!

Map the lost-vegetation mask:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.pcolormesh(lost_vegetation.longitude, lost_vegetation.latitude, lost_vegetation,
              cmap=mcolors.ListedColormap(["#eeeeee", "firebrick"]))
ax.scatter(122.95, 10.68, color="black", s=30, zorder=5)
ax.set_title("Lost vegetation, 1996 -> 2023 (our own classifier)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

frac_lost_ours = float(lost_vegetation.sum()) / int((landcover_1996_calibrated == "Vegetation").sum())
print(f"Fraction of 1996's Vegetation-classified pixels that changed by 2023: {frac_lost_ours * 100:.1f}%")

print(f"\nFor comparison -- Hansen's independent full 2001-2023 forest-loss record (subsection 1):")
print(f"  Our classifier: {frac_lost_ours * 100:.1f}%   Hansen (2001-2023, vs. 2000 baseline): {frac_lost_hansen_full * 100:.1f}%")

> 💬 **Discussion:** Our own classifier says roughly **2.7%** of 1996's vegetated pixels changed by 2023 — in the same ballpark as Hansen's independent **~3.5%** forest-loss estimate over a comparable era. They're not identical: our "Vegetation" class lumps forest together with farmland/grass, while Hansen's `lossyear` specifically tracks forest; and the two records span slightly different windows. But landing within the same order of magnitude — instead of being off by 15x, the way a raw single-date snapshot comparison would have been — is exactly what properly-engineered composite data plus a robust (NDVI-based) rule buys you. Same three-line classifier, same `detect_change` function: the only thing that changed was the *input data's* quality.

---

> 💬 **Final discussion:**
> - Look back at Part II's Philippine Eagle map and the EN/CR/VU mammal species you found on Negros. Does the forest-loss pattern you just mapped and quantified help explain why those species are threatened?
> - Across all of Day 2, you built **two** homemade classifiers from raw data — one for climate (Part I), one for land cover (Part III) — validated both against independent "official" products (Beck et al., Hansen), and used the second one to directly measure real-world change over time. That loop (rule -> function -> grid -> map -> validate) is the core pattern behind an enormous amount of real environmental data science.

---

## Well done — Day 2 complete!

Across today you:
- Learned `def` functions and `for` loops (including nested loops), and used them to build a full climate classification pipeline from scratch
- Classified the entire Philippines at three points in time, and validated your homemade model against Beck et al. (2023)
- Explored forest cover, satellite imagery, and real (messy!) biodiversity data for Negros
- Built a second homemade classifier — this time for land cover — reusing the exact same rule -> function -> grid -> map pattern from Part I
- Used ocean pixels as a physical reference point to calibrate brightness between two different satellite sensors (Landsat 5 and Sentinel-2) — and saw why that fix helps a lot without being perfect
- Measured real land-use change on Negros using your own classifier, cross-checked against Hansen et al.'s independent forest-loss product
- Connected climate, vegetation, satellite imagery, biodiversity, and land-use change into a single, coherent story about one island

Tomorrow: Day 3.